# GitHub API

_Part of the **Decay Before Archival** project — see `README.md` for full project context, setup instructions, and links to the other notebooks (`deps_dev.ipynb`, `scorecard.ipynb`, `gh_archive.ipynb`, `pypi.ipynb`, `github_api.ipynb`, `other_sources.ipynb`)._

In [ ]:
import os
import time

import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv()  # picks up GITHUB_TOKEN from .env if you have one (optional, see below)

session = requests.Session()
session.headers.update({
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
    "User-Agent": "decay-before-archival",  # required by the GitHub API
})

token = os.environ.get("GITHUB_TOKEN")  # optional: raises rate limits a lot (5000 req/h vs 60)
if token:
    session.headers["Authorization"] = f"Bearer {token}"

print("authenticated" if token else "unauthenticated — search is limited to 10 requests/min, core API to 60/hour")

## Explore: GitHub Search API

The GitHub REST API fills gaps the BigQuery sources don't cover directly — live repo state (stars, forks, open issues/PRs, archived flag, last push) for repos that may not be in deps.dev/Scorecard yet.

**Planned analysis:** build the candidate pool of **open-source repos with 1000-5000 stars created between 2024 and 2025** (`stars:1000..5000 created:2024-01-01..2025-12-31`). GitHub's search API has no license qualifier, so the open-source filter is applied after fetching — keep only repos with an identifiable SPDX license (`license_spdx`); `NOASSERTION` (custom license) and unlicensed repos are dropped by default. These are young-but-popular projects — the cohort we'll later watch for decay signals (stalled releases, contributor turnover, issue backlog growth) before archival.

**Rate limits:** unauthenticated search allows 10 requests/min and the core API 60/hour; a `GITHUB_TOKEN` in `.env` (free account) raises that to 30 searches/min and 5,000 core req/h. The `gh_get` helper below backs off automatically when it hits a limit.

In [ ]:
# Headers of the most recent response — fetch_all() reads the Link header from here for pagination.
_last_headers = {}

def gh_get(url, params=None):
    """GET with automatic backoff on rate-limit (403/429). Returns parsed JSON."""
    if "session" not in globals():
        raise RuntimeError(
            "GitHub session is not set up — run the first code cell "
            "(requests.Session + GITHUB_TOKEN) before running this notebook's queries."
        )
    while True:
        resp = session.get(url, params=params, timeout=30)
        if resp.status_code in (403, 429):
            reset = int(resp.headers.get("X-RateLimit-Reset", time.time() + 60))
            wait = max(reset - time.time(), 1) + 1
            print(f"rate-limited — sleeping {wait:.0f}s")
            time.sleep(wait)
            continue
        resp.raise_for_status()
        _last_headers.clear()
        _last_headers.update(resp.headers)
        return resp.json()


In [ ]:
# Candidate pool: repos created 2024-2025 with 1000-5000 stars, most-starred first
query = "stars:1000..5000 created:2024-01-01..2025-12-31"
per_page = 100
max_pages = 3  # cap while exploring; raise once we know how many we actually want

items, page = [], 1
while page <= max_pages:
    data = gh_get("https://api.github.com/search/repositories", params={
        "q": query, "sort": "stars", "order": "desc",
        "per_page": per_page, "page": page,
    })
    print(f"total matching repos: {data['total_count']:,} — fetched page {page}")
    items.extend(data["items"])
    if data.get("incomplete_results"):
        print("warning: GitHub flagged these results as incomplete (query too broad) — narrow the query")
    if len(data["items"]) < per_page:
        break
    page += 1

repos = pd.DataFrame(items)
repos["license_spdx"] = repos["license"].apply(lambda L: L["spdx_id"] if L else None)
repos = repos[[
    "full_name", "stargazers_count", "forks_count", "open_issues_count",
    "language", "created_at", "pushed_at", "archived", "license_spdx", "description",
]].rename(columns={"full_name": "repo_full_name"})
repos["repo_owner"] = repos["repo_full_name"].str.split("/").str[0]
repos["repo_name"] = repos["repo_full_name"].str.split("/").str[1]
repos["created_at"] = pd.to_datetime(repos["created_at"]).dt.date
repos["pushed_at"] = pd.to_datetime(repos["pushed_at"]).dt.date

# GitHub's search API has no license qualifier, so "open source" is filtered here:
#   license_spdx == None        -> no license file at all (all rights reserved — not open source)
#   license_spdx == "NOASSERTION" -> custom/unrecognized license (may still be OSS — check manually)
OPEN_SOURCE_ONLY = True
if OPEN_SOURCE_ONLY:
    n_before = len(repos)
    repos = repos[repos["license_spdx"].notna()]
    print(f"open-source filter: {n_before} -> {len(repos)} rows "
          f"(dropped {n_before - len(repos)} without an identifiable license)")

repos.head(20)

**Notes:**

_(add observations here)_

In [ ]:
# Quick look at the pool: size, year split, languages, and how many are already archived
repos["created_year"] = pd.to_datetime(repos["created_at"]).dt.year
print(f"rows fetched: {len(repos):,}")
print("\nby creation year:")
print(repos["created_year"].value_counts().sort_index())
print("\ntop languages:")
print(repos["language"].value_counts(dropna=False).head(8))
print(f"\nalready archived: {int(repos['archived'].sum())}")

**Notes:**

_(add observations here)_

In [ ]:
# Per-repo detail for one sample repo: contributors + open issues (bus factor, issue backlog)
sample_repo = repos.iloc[0]["repo_full_name"]  # most-starred in the pool; swap in any "owner/name"
print(f"exploring {sample_repo}")

# Contributors (top page of 100 by commit count)
contributors = gh_get(f"https://api.github.com/repos/{sample_repo}/contributors", params={"per_page": 100})
contrib_df = pd.DataFrame(contributors)[["login", "contributions"]]

# Open issues (includes PRs — GitHub counts them together in open_issues_count)
issues = gh_get(f"https://api.github.com/repos/{sample_repo}/issues", params={"per_page": 100, "state": "open"})
issue_df = pd.DataFrame(issues)[["number", "title", "created_at", "pull_request"]]
issue_df["is_pr"] = issue_df["pull_request"].notna()

total_commits = contrib_df["contributions"].sum()
top_share = contrib_df.iloc[0]["contributions"] / total_commits if total_commits else float("nan")
print(f"contributors (top {len(contrib_df)}): {len(contrib_df)}, top contributor share: {top_share:.1%}")
print(f"open issues/PRs (first 100): {len(issue_df)}, of which PRs: {int(issue_df['is_pr'].sum())}")
contrib_df.head(10)

**Notes:**

_(add observations here)_

## Collecting the Data Collection Plan metrics

The [Data Collection Plan](../Data%20Collection%20Plan%20Template.csv) assigns these metrics to the **GitHub API** source; this section collects as many of them as the REST API allows, per repo in the pool above:

| Plan metric | How it's collected here |
|---|---|
| Merged PR counts | `GET /repos/{repo}/pulls?state=closed` — count closed PRs with a non-null `merged_at`, in the trailing window (`since`) |
| Opened Issues | `GET /repos/{repo}/issues?state=all&since=` — items without a `pull_request` key, created in the window (excludes PRs) |
| Opened PRs | same call — items **with** a `pull_request` key, created in the window |
| Resolved Issues | `GET /repos/{repo}/issues?state=closed&since=` — closed issues without a `pull_request` key (PR merges counted separately) |
| Time Since last Commit | `GET /repos/{repo}/commits?per_page=1` (default branch, newest first) — days between that commit and today. Note: counts bot commits too; the plan says *human* commits — flag for refinement |
| Time Since last release | `GET /repos/{repo}/releases?per_page=1` — days since the latest published release (empty list → never released) |
| Bus Factor | `GET /repos/{repo}/contributors?per_page=100` — smallest number of top contributors whose combined commits reach 50% of total. Capped at the first 100 contributors; bot accounts (`[bot]`) are excluded to approximate *human* contributors |

**Also captured for free from the pool query:** `Stars` and `Forks` (current totals — the plan's *new during window* version needs a time series, see below), plus `open_issues_count`, `archived`, `license_spdx` as context columns.

**Not collectible from the GitHub API** (other sources own these): Commit Frequency (deps.dev), Downloads (PyPI/npm registries), Criticality/Scorecard/Maintained/Code-Review/CI-Tests scores (OpenSSF Scorecard), Number of dependent projects (deps.dev).

**Window:** the plan says "during the observation period" — `WINDOW_DAYS` below sets a trailing window for all rate-based counts (opened/closed issues, merged PRs). Set it to match whatever the group decides the observation period is. One API quirk handled in the code: GitHub's `since` parameter filters by *update* time, not creation time, so it only narrows the candidate set — every count is re-filtered on the client against the actual `created_at` / `closed_at` / `merged_at` timestamps.

**Scale & rate limits:** each repo costs ~6 core API calls. The full pool of ~259 repos ≈ 1,600 calls — that's hours unauthenticated (60/h) or a few minutes with a `GITHUB_TOKEN` (5,000/h). Progress is checkpointed to `data/github_metrics.csv` after every repo, so if you get rate-limited mid-run, re-running the loop just resumes where it left off. For exploration, set `N_SAMPLE` below to a small number first.

In [ ]:
import datetime as dt

# Observation window for rate-based metrics (plan: "during the observation period")
WINDOW_DAYS = 90
since = (dt.date.today() - dt.timedelta(days=WINDOW_DAYS)).isoformat()

# How many repos to collect this run — small while exploring; use len(repos) for the full pool.
# (Resumable: rows already in data/github_metrics.csv are skipped, so you can raise this over several runs.)
N_SAMPLE = 8

print(f"window: last {WINDOW_DAYS} days (since {since}) — collecting for up to {N_SAMPLE} repos")

In [ ]:
import os
from pathlib import Path

# Anchor the checkpoint file to the repo root (works whether the kernel's cwd is the
# repo root — JupyterLab — or notebooks/ — jupyter nbconvert).
def _repo_root():
    for cand in [Path.cwd(), *Path.cwd().parents]:
        if (cand / "pyproject.toml").exists():
            return cand
    return Path.cwd()

DATA_DIR = _repo_root() / "data"
METRICS_CSV = str(DATA_DIR / "github_metrics.csv")
os.makedirs(DATA_DIR, exist_ok=True)

def fetch_all(url, params=None):
    """Follow every page of a list endpoint (Link header)."""
    params = dict(params or {})
    params.setdefault("per_page", 100)
    out, page = [], 1
    while True:
        data = gh_get(url, params={**params, "page": page})
        out.extend(data)
        link = _last_headers.get("Link", "")
        if 'rel="next"' not in link:
            break
        page += 1
    return out

In [ ]:
def collect_repo(full_name):
    """Collect the plan's GitHub-API metrics for one repo. ~6 API calls.

    Note: GitHub's `since` param filters by *update* time, not creation time — it only
    narrows the candidate set (a superset), so every count below is re-filtered on the
    client against the actual created_at / closed_at / merged_at timestamps.
    """
    row = {"repo_full_name": full_name}

    # --- Issues + PRs opened in window (one call: state=all since=window start) ---
    opened = fetch_all(f"https://api.github.com/repos/{full_name}/issues",
                       params={"state": "all", "since": since})
    created_in_window = [i for i in opened if pd.to_datetime(i["created_at"]).date() >= dt.date.fromisoformat(since)]
    row["opened_issues"] = sum(1 for i in created_in_window if not i.get("pull_request"))
    row["opened_prs"] = sum(1 for i in created_in_window if i.get("pull_request"))

    # --- Resolved (closed) issues in window: closed_at inside the window, PRs excluded ---
    closed = fetch_all(f"https://api.github.com/repos/{full_name}/issues",
                       params={"state": "closed", "since": since})
    row["resolved_issues"] = sum(
        1 for i in closed
        if not i.get("pull_request") and pd.to_datetime(i["closed_at"]).date() >= dt.date.fromisoformat(since)
    )

    # --- Merged PRs in window: merged_at inside the window ---
    prs = fetch_all(f"https://api.github.com/repos/{full_name}/pulls",
                    params={"state": "closed", "since": since})
    row["merged_prs"] = sum(
        1 for p in prs if p.get("merged_at") and pd.to_datetime(p["merged_at"]).date() >= dt.date.fromisoformat(since)
    )

    # --- Time since last commit (newest commit on default branch) ---
    commits = gh_get(f"https://api.github.com/repos/{full_name}/commits", params={"per_page": 1})
    if commits:
        row["last_commit_at"] = pd.to_datetime(commits[0]["commit"]["committer"]["date"]).date()
        row["days_since_last_commit"] = (dt.date.today() - row["last_commit_at"]).days
    else:
        row["last_commit_at"] = None
        row["days_since_last_commit"] = None

    # --- Time since last release (empty list -> never released) ---
    releases = gh_get(f"https://api.github.com/repos/{full_name}/releases", params={"per_page": 1})
    if releases:
        row["last_release_at"] = pd.to_datetime(releases[0]["published_at"]).date()
        row["days_since_last_release"] = (dt.date.today() - row["last_release_at"]).days
    else:
        row["last_release_at"] = None
        row["days_since_last_release"] = None

    # --- Bus factor: min contributors covering >=50% of commits (top 100, bots excluded) ---
    contribs = gh_get(f"https://api.github.com/repos/{full_name}/contributors", params={"per_page": 100})
    contribs = [c for c in contribs if not c["login"].endswith("[bot]")]
    total = sum(c["contributions"] for c in contribs)
    bus, acc = None, 0
    for n, c in enumerate(contribs, start=1):  # already sorted by contributions desc
        acc += c["contributions"]
        if total and acc * 2 >= total:
            bus = n
            break
    row["bus_factor"] = bus
    row["n_contributors_top100"] = len(contribs)
    return row

In [ ]:
# Batch-collect the pool, resuming from data/github_metrics.csv if it exists
target = repos.head(N_SAMPLE)  # most-starred first; swap for a random sample later if desired

if os.path.exists(METRICS_CSV):
    done = set(pd.read_csv(METRICS_CSV)["repo_full_name"])
else:
    done = set()

pending = target[~target["repo_full_name"].isin(done)]
print(f"collecting {len(pending)} repos ({len(done)} already in {METRICS_CSV})")

def save_row(row):
    """Append one repo's metrics immediately — a rate-limit mid-run never loses finished work."""
    chunk = pd.DataFrame([row])
    old = pd.read_csv(METRICS_CSV) if os.path.exists(METRICS_CSV) else pd.DataFrame()
    pd.concat([old, chunk], ignore_index=True).to_csv(METRICS_CSV, index=False)

t0 = time.time()
for i, r in enumerate(pending.itertuples(index=False), start=1):
    try:
        save_row(collect_repo(r.repo_full_name))
    except Exception as e:  # a single repo failing (e.g. API glitch) shouldn't kill the run
        print(f"  ! {r.repo_full_name}: {type(e).__name__}: {e}")
    if i % 5 == 0 or i == len(pending):
        rate = i / (time.time() - t0) if time.time() > t0 else float("nan")
        print(f"  {i}/{len(pending)} done ({rate:.2f} repos/s, checkpointed to {METRICS_CSV})")

print(f"finished: {len(pd.read_csv(METRICS_CSV)) if os.path.exists(METRICS_CSV) else 0} rows total in {METRICS_CSV}")

In [ ]:
# Merge collected metrics back onto the pool and inspect
metrics = pd.read_csv(METRICS_CSV)
for col in ["last_commit_at", "last_release_at"]:
    metrics[col] = pd.to_datetime(metrics[col]).dt.date

combined = repos.merge(
    metrics[["repo_full_name", "opened_issues", "opened_prs", "resolved_issues",
             "merged_prs", "days_since_last_commit", "last_commit_at",
             "days_since_last_release", "last_release_at",
             "bus_factor", "n_contributors_top100"]],
    on="repo_full_name", how="left")

print(f"pool: {len(repos)} rows, with collected metrics: {combined['bus_factor'].notna().sum()}")

# decay-at-a-glance view: quiet repos (stale commits/releases) and fragile ones (low bus factor)
combined[[
    "repo_full_name", "stargazers_count", "days_since_last_commit", "days_since_last_release",
    "bus_factor", "opened_issues", "resolved_issues", "merged_prs", "archived",
]].sort_values("days_since_last_commit", ascending=False).head(20)

**Notes:**

_(add observations here)_

## Join key for integration

`repo_owner` + `repo_name` (and the full `full_name`/`html_url`) — same owner/name pair used by GH Archive (`payload.repo.owner.name` / `name`) and OpenSSF Scorecard, so this source joins directly on those. Package-based sources (deps.dev, PyPI) join via the repo's package manifest or a name match in the integration step.

**Export shape:** the pool table (`repos`) plus `data/github_metrics.csv` (one row per repo: `repo_full_name`, opened/closed issue & PR counts, merged PRs, days since last commit/release, bus factor) — merge on `repo_full_name` in `integrate_datasets.ipynb`.